## 📌 **Informe de Laboratorio**
  ###  Medicion del τ de un capacitor

        Laboratorio Avanzado I


### 👥 **Integrantes del grupo:**

- Cristian Vergara
- Francesco Luligo
- Juan Jose Palacio
- Sofia Moscoso

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## **Resumen**
En esta actividad, se midio el tiempo de vida media de un capacitor de 10e-7 faradios, mediante un circuito sencillo en el que usamos dos resistencias de 33kiloohmios. Tuvimos como objetivo aprender acerca del uso de un microcontrolador ESP32 como procesador para automatizar procesos de medicion. Usamos las plataformas: Mu, la cual implementa la accion en el procesador del microcontrolador, realizando la conexion serial para manipular los datos en Phyton.

## **Introducción**
La carga y descarga de un capacitor constituyen procesos fundamentales en el estudio de los circuitos eléctricos. Un capacitor almacena energía en forma de campo eléctrico cuando se conecta a una fuente de tensión, acumulando una carga proporcional al voltaje aplicado. Este proceso de carga sigue una ley exponencial descrita por la ecuación $$ V(t) = V_0 (1 - e^{-t/RC}) $$, donde ( R ) es la resistencia del circuito y ( C ) la capacitancia. De manera análoga, durante la descarga, el voltaje en los bornes del capacitor decrece exponencialmente según $$ V(t) = V_0 e^{-t/RC} $$. El producto ( RC ) se conoce como la constante de tiempo del circuito y representa el tiempo característico en el cual el capacitor alcanza aproximadamente el 63% de su carga o descarga total. Este parámetro es esencial para comprender el comportamiento dinámico de sistemas eléctricos y electrónicos.

Para la automatización del proceso de medición de este fenómeno, se empleó el microcontrolador **ESP32**, un dispositivo de bajo costo y alta eficiencia que integra un procesador de doble núcleo, módulos de conectividad Wi-Fi y Bluetooth, y múltiples interfaces de entrada y salida analógicas y digitales. Gracias a estas características, el ESP32 resulta ideal para realizar tareas de adquisición de datos, control de dispositivos y comunicación con otros sistemas. Su versatilidad lo convierte en una herramienta ampliamente utilizada en proyectos de instrumentación, robótica y monitoreo en tiempo real.

La **automatización de los procesos de medición** en electrónica permite obtener datos más precisos, repetibles y confiables, reduciendo el error humano y optimizando el tiempo de adquisición. Además, facilita la recopilación de grandes volúmenes de información que pueden ser analizados posteriormente mediante software especializado. En este contexto, la integración entre hardware y software es esencial para el desarrollo de sistemas modernos de medición y control.

Para implementar este proyecto, se utilizó la plataforma **Mu**, un entorno de desarrollo sencillo que facilita la programación y comunicación con el microcontrolador ESP32. A través de la conexión serial, los datos adquiridos por el microcontrolador fueron enviados y procesados en **Python**, un lenguaje de programación ampliamente empleado por su versatilidad y capacidad para el análisis de datos, visualización y control de dispositivos. Esta combinación permitió automatizar el registro de voltajes durante la carga y descarga del capacitor, posibilitando así la determinación experimental de su tiempo de vida media de manera precisa y eficiente.


## **Procedimiento:**  
1. Calculamos el valor teorico esperado del tiempo de vida media del capacitor, el cual tuvo un resultado de 6.6ms, con el fin de comparar posteriormente con el resulado experimental.

2. Construimos el codigo en microphyton(MU), el cual, configura el microcontrolador ESP32 para medir señales analógicas a través de un pin ADC, almacenar los datos con su respectivo tiempo y enviarlos al computador cuando el usuario lo indique. Además, usa un LED y un pin de control (“pulso”) como indicadores del estado del proceso de medición.

3. Construimos el codigo en Phyton que nos permite establecer una comunicación serial entre el computador y el ESP32, enviar el comando que inicia la medición ('a'), recibir los datos generados por el microcontrolador (tiempo y voltaje), almacenarlos en listas y finalmente graficarlos usando la biblioteca matplotlib.

4. Analizamos los datos obtenidos mediante la comunicacion serial al visualizarlos en la grafica, comparando con los resultados esperados teoricos. Realizamos una interpolacion para obtener el valor exacto correspondiente al tiempo de vida media del capacitor y, finalmente, concluimos con el porcentaje de error obtenido en la practica experimental.


##Codigo en MU
Codigo que describe las "acciones" que realizara el microcontrolador ESP32 sobre el circuito sencillo que contiene las resistencias y el capacitor, ademas, define las variables que seran medidas y almacenadas en el espacio de memoria del ESP32. Esta diseñado de modo que el programa inicie cuando se lo indique el usuario, tenga como salida un voltaje de 3,3V, tome medidas en un determinado intervalo de tiempo que sera escogido segun los parametros del sistema, en nuestro caso, este fue de 0.8ms, y termine al tomar una cantidad N de datos. Esta realizado para MU. no funciona en este colab.

In [ ]:
from machine import Pin, ADC
import time
import sys
import select

In [ ]:

led = Pin(2, Pin.OUT)
pulso = Pin(14, Pin.OUT)
adc = ADC(Pin(4))
adc.atten(ADC.ATTN_11DB)
adc.width(ADC.WIDTH_12BIT)

N = 150
datos = []
retardo = 8e-4

print("envíe 'a' para iniciar")

while True:
    if sys.stdin in select.select([sys.stdin], [], [], 0)[0]:
        c = sys.stdin.read(1)
        if c == 'a':
            led.on()
            print('midiendo...')
            pulso.off()
            tiempoantes = time.ticks_us()
            for i in range(N):
                dato = adc.read()
                tiempoactual = time.ticks_diff(time.ticks_us(), tiempoantes) / 1e6
                datos.append((tiempoactual, dato))
                time.sleep(retardo)
            print('medición terminada, enviandodatos')
            for (tiempoactual, dato) in datos:
                print('{:.6f}, {}'.format(tiempoactual, dato))
            print('FIN')
        else:
            pulso.on()
            led.off()
    else:
        pulso.on()
        led.off()
        time.sleep(retardo)

##Codigo en Py
En phyton, manipulamos los datos tomados por el microcontrolador para realizar el debido analisis. Esto se realiza mediante la carpeta serial, la cual, conecta el microcontrolador con phyton. Observamos un coportamiento esperado en los datos experimentales, segun
$$
V_C(t) = V_0\left(e^{-\frac{t}{RC}}\right)
$$
Esta realizado en microphyton. No funciona en el colab, no esta instalado Serial.


In [ ]:
import serial

In [ ]:
puerto = 'COM3'
baudrate = 115200

ser = serial.Serial(puerto, baudrate, timeout=1)

print('conectando al puerto', puerto)

ser.reset_input_buffer()
ser.write(b'a')

print("comando 'a' enviado")

tiempos = []
voltajes = []

print('recibiendo datos...\n')

while True:
    linea = ser.readline().decode().strip()

    if not linea:
        continue

    if linea == 'FIN':
        print('transmision finalizada')
        break

    try:
        t_str,v_str = linea.split(',')
        tiempos += [float(t_str)]
        voltajes += [float(v_str)]
        print(f'tiempo: {t_str} s, voltaje: {v_str} mV' )

    except ValueError:
        continue

plt.plot(tiempos,voltajes)
plt.title('Medicion ADC ESP32')
plt.xlabel('Tiempo s')
plt.ylabel('Lectura ADC (,4095)')
plt.grid(True)
plt.show()

ser.close()

In [ ]:
datos = pd.read_csv('https://raw.githubusercontent.com/SoffMoscoso/PhysicsLabs/refs/heads/main/Advanced%20laboratory%20I/Data/datos_adc.csv')
t = np.linspace(0,0.015,10)
R,C = 66000,1e-7
v = 3300*np.exp(- t/ (R*C))

plt.plot(datos["Tiempo (s)"], datos["Voltaje (mV)"], 'o', label='datos experimentales')
plt.plot(t, v, '-',c ="red", label='datos teoricos')
plt.title('V vs t')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.show()

##Datos e Interpolacion.
Para hallar el tiempo de vida medio del capacitor, realizamos una interpolacion de los datos obtenidos con la intencion de tomar el valor del tiempo en el valor de voltaje exacto correspondiente (V0/e = 1214). De modo que, para ello, invertimos los ejes con los datos que teniamos, para expresar el voltaje V como la variable independiente y el tiempo en funcion del voltaje. La razon por la cual realizar el procedimiento de inversion es valido esta justificado con la biyectividad de la funcion.

In [ ]:
datos

##

In [ ]:
# Puntos conocidos (x, y)
x = np.array(datos["Voltaje (mV)"])
y = np.array(datos["Tiempo (s)"])

# Ajuste de polinomio de grado 2 (cuadrático)
coeficientes = np.polyfit(x, y,8)

# Crear función polinómica
p = np.poly1d(coeficientes)

# Evaluar en un rango para graficar
x_interp = np.linspace(min(x), max(x), 100)
y_interp = p(x_interp)

# Graficar
plt.plot(x, y, 'ro', label='Puntos conocidos')
plt.plot(x_interp, y_interp, 'b-', label='Interpolación cuadrática')
plt.title('Interpolación Cuadrática')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
Valor_exp =p(1214)
Valor_teo = 6.6e-3

##Conclusion

In [ ]:
error = abs(Valor_exp - Valor_teo)/Valor_teo

print(f"""El valor teorico esperado es de 6.6ms, sin embargo, el resultado experimental es de {round((p(x[0]/np.exp(1)) )*1000 , 2)} ms.
 #La medicion tuvo entonces un error del {round(error*100,2)}% que corresponden a  las incertidumbres del 10%
 de cada una de las resistencias y del capacitor.
 """)